###**Silver layer**

###Enable Change Data Feed

In [0]:
%sql
-- Enables Change Data Feed (CDF) for all Silver tables.
ALTER TABLE edtech.silver.blogs            SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE edtech.silver.newsletters      SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE edtech.silver.coursera_courses SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE edtech.silver.microsoft_learn  SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE edtech.silver.github_repos     SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE edtech.silver.youtube_videos   SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

### Set Time Parser Policy


In [0]:
-- Enables legacy date/time parsing.
SET spark.sql.legacy.timeParserPolicy = LEGACY;

key,value
spark.sql.legacy.timeParserPolicy,LEGACY


### Load Blogs into Silver

This cell cleans, standardizes, deduplicates, and merges blog data from Bronze into Silver.

In [0]:
%sql

-- Cleans, standardizes, deduplicates, and updates blog data from Bronze to Silver.

MERGE INTO edtech.silver.blogs AS t

USING (
  WITH c AS (
    SELECT
      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,
      TRIM(`list of keywords`) AS list_of_keywords,

      'English' AS language,

      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,

      COALESCE(
        TRY_CAST(published_date AS TIMESTAMP),
        TRY_TO_TIMESTAMP(
          REGEXP_REPLACE(
            TRIM(published_date),
            ' (GMT|UTC|UT|Z)$',
            ' +0000'
          ),
          'EEE, d MMM yyyy HH:mm:ss Z'
        )
      ) AS published_date,

      COALESCE(
        TRY_CAST(last_updated AS TIMESTAMP),
        TRY_TO_TIMESTAMP(
          REGEXP_REPLACE(
            TRIM(last_updated),
            ' (GMT|UTC|UT|Z)$',
            ' +0000'
          ),
          'EEE, d MMM yyyy HH:mm:ss Z'
        )
      ) AS last_updated,

      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.rss_raw

    WHERE LOWER(TRIM(content_type)) = 'article'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
    OR (t.published_date IS NULL AND s.published_date IS NOT NULL)
) THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;


-- Standardizes existing language values to English.

UPDATE edtech.silver.blogs
SET language = 'English'
WHERE LOWER(TRIM(language)) = 'english';

num_affected_rows
565


### Check Bronze Blog Data

This cell counts article records and missing publication dates in Bronze.

In [0]:
%sql
-- Counts article records and missing publication dates in Bronze.

SELECT COUNT(*) AS bronze_rows,
       SUM(CASE WHEN published_date IS NULL OR TRIM(published_date) = '' THEN 1 ELSE 0 END) AS bronze_null_dates
FROM edtech.bronze.rss_raw
WHERE LOWER(TRIM(content_type)) = 'article';

bronze_rows,bronze_null_dates
565,0


### Load Newsletters into Silver

This cell cleans, standardizes, deduplicates, and updates newsletter data from Bronze to Silver.

In [0]:
%sql

-- Cleans, standardizes, deduplicates, and updates newsletter data from Bronze to Silver.

MERGE INTO edtech.silver.newsletters AS t

USING (
  WITH c AS (
    SELECT
      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,
      TRIM(`list of keywords`) AS list_of_keywords,
      'English' AS language,
      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,

      COALESCE(
        TRY_CAST(published_date AS TIMESTAMP),
        TRY_TO_TIMESTAMP(
          REGEXP_REPLACE(
            REGEXP_REPLACE(TRIM(published_date), '^[A-Za-z]+, *', ''),
            ' +(GMT|UTC|UT|Z)$',
            ' +0000'
          ),
          'd MMM yyyy HH:mm:ss Z'
        )
      ) AS published_date,

      COALESCE(
        TRY_CAST(last_updated AS TIMESTAMP),
        TRY_TO_TIMESTAMP(
          REGEXP_REPLACE(
            REGEXP_REPLACE(TRIM(last_updated), '^[A-Za-z]+, *', ''),
            ' +(GMT|UTC|UT|Z)$',
            ' +0000'
          ),
          'd MMM yyyy HH:mm:ss Z'
        )
      ) AS last_updated,

      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.rss_raw

    WHERE LOWER(TRIM(content_type)) = 'newsletter'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
    OR (t.published_date IS NULL AND s.published_date IS NOT NULL)
) THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;


-- Standardizes existing language values to English.

UPDATE edtech.silver.newsletters
SET language = 'English'
WHERE LOWER(TRIM(language)) = 'english';

num_affected_rows
35


### Load Coursera Courses into Silver

This cell cleans, standardizes, deduplicates, and updates Coursera course data from Bronze to Silver.

In [0]:
-- Cleans, standardizes, deduplicates, and updates Coursera course data from Bronze to Silver.

MERGE INTO edtech.silver.coursera_courses AS t
USING (
  WITH c AS (
    SELECT
      NULLIF(TRIM(content_id), '')          AS content_id,
      NULLIF(TRIM(title), '')               AS title,
      TRIM(description)                     AS description,
      CASE
       WHEN LOWER(TRIM(topic)) = 'ai'    THEN 'AI'
       WHEN LOWER(TRIM(topic)) = 'data'  THEN 'Data'
       WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
       ELSE INITCAP(TRIM(topic))
      END AS topic,
      TRIM(category)                        AS category,
      TRIM(`list of keywords`)              AS list_of_keywords,
      TRIM(language)                        AS language,
      TRIM(source)                          AS source,
      NULLIF(TRIM(url), '')                 AS url,
      TRIM(content_type)                    AS content_type,
      INITCAP(NULLIF(TRIM(difficulty_level), '')) AS difficulty_level,
      TRY_CAST(published_date AS TIMESTAMP) AS published_date,
      TRY_CAST(last_updated   AS TIMESTAMP) AS last_updated,
      current_timestamp()                   AS silver_ingested_at
    FROM edtech.bronze.api_raw
    WHERE LOWER(TRIM(source)) = 'coursera'
  )
  SELECT * FROM c
  WHERE content_id IS NOT NULL AND title IS NOT NULL AND url IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id ORDER BY last_updated DESC NULLS LAST) = 1
) AS s
ON t.content_id = s.content_id
WHEN MATCHED AND (s.last_updated > t.last_updated
              OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL))
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### Load Microsoft Learn into Silver

This cell cleans, standardizes, generates keywords, deduplicates, and updates Microsoft Learn data from Bronze to Silver.

In [0]:
-- Cleans, standardizes, generates keywords, deduplicates, and updates Microsoft Learn data from Bronze to Silver.

MERGE INTO edtech.silver.microsoft_learn AS t

USING (

  WITH c AS (

    SELECT

      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,

      -- 3-tier keyword extraction: dictionary → title & description → category
      COALESCE(

        -- Tier 1: Dictionary matching
        NULLIF(CONCAT_WS(', ',

          CASE WHEN LOWER(title || ' ' || description) LIKE '%artificial intelligence%' THEN 'artificial intelligence' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%machine learning%' THEN 'machine learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%deep learning%' THEN 'deep learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%generative ai%' THEN 'generative ai' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%large language model%' THEN 'large language model' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%llm%' THEN 'llm' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%computer vision%' THEN 'computer vision' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%nlp%' THEN 'nlp' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%reinforcement learning%' THEN 'reinforcement learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%transformer%' THEN 'transformer' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%neural network%' THEN 'neural network' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%gpt%' THEN 'gpt' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%pytorch%' THEN 'pytorch' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%tensorflow%' THEN 'tensorflow' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%data engineering%' THEN 'data engineering' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data pipeline%' THEN 'data pipeline' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data warehouse%' THEN 'data warehouse' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data lake%' THEN 'data lake' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%etl%' THEN 'etl' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%spark%' THEN 'apache spark' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kafka%' THEN 'apache kafka' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%sql%' THEN 'sql' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%database%' THEN 'database' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%big data%' THEN 'big data' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%databricks%' THEN 'databricks' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%postgresql%' THEN 'postgresql' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud computing%' THEN 'cloud computing' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud architecture%' THEN 'cloud architecture' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud security%' THEN 'cloud security' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kubernetes%' THEN 'kubernetes' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%serverless%' THEN 'serverless' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%terraform%' THEN 'terraform' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%devops%' THEN 'devops' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%microservices%' THEN 'microservices' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%aws%' THEN 'aws' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%azure%' THEN 'azure' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%google cloud%' THEN 'google cloud' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%docker%' THEN 'docker' END

        ), ''),

        -- Tier 2: Extract keywords from title and description
        NULLIF(
          LOWER(
            CONCAT_WS(', ',
              SLICE(
                FILTER(
                  TRANSFORM(
                    SPLIT(
                      LOWER(CONCAT_WS(' ', title, description)),
                      ' '
                    ),
                    w -> REGEXP_REPLACE(w, '[^a-z0-9]', '')
                  ),
                  w -> LENGTH(w) >= 4
                       AND w NOT IN (
                         'with','from','this','that','your','have',
                         'what','when','where','which','will','into',
                         'over','using','introduction','course','guide',
                         'tutorial','learn','basics','advanced','complete',
                         'master','part','step','about','they','their',
                         'there','these','those','then','than','also',
                         'more','some'
                       )
                ),
                1,
                5
              )
            )
          ),
          ''
        ),

        -- Tier 3: Category fallback
        LOWER(TRIM(category))

      ) AS list_of_keywords,

      TRIM(language) AS language,
      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,

      INITCAP(
        NULLIF(TRIM(difficulty_level), '')
      ) AS difficulty_level,

      TRY_CAST(published_date AS TIMESTAMP) AS published_date,
      TRY_CAST(last_updated AS TIMESTAMP) AS last_updated,

      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.api_raw

    WHERE LOWER(TRIM(source)) = 'microsoft learn'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
)

THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### Load GitHub Repos into Silver

This cell cleans, standardizes, generates keywords, deduplicates, and updates GitHub data from Bronze to Silver.

In [0]:
-- Cleans, standardizes, generates keywords, deduplicates, and updates GitHub data from Bronze to Silver.

MERGE INTO edtech.silver.github_repos AS t

USING (

  WITH c AS (

    SELECT

      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,

      -- 3-tier keyword extraction: dictionary → title & description → category
      COALESCE(

        -- Tier 1: Dictionary matching
        NULLIF(CONCAT_WS(', ',

          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%artificial intelligence%' THEN 'artificial intelligence' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%machine learning%' THEN 'machine learning' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%deep learning%' THEN 'deep learning' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%generative ai%' THEN 'generative ai' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%large language model%' THEN 'large language model' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%llm%' THEN 'llm' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%computer vision%' THEN 'computer vision' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%nlp%' THEN 'nlp' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%reinforcement learning%' THEN 'reinforcement learning' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%transformer%' THEN 'transformer' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%neural network%' THEN 'neural network' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%gpt%' THEN 'gpt' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%pytorch%' THEN 'pytorch' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%tensorflow%' THEN 'tensorflow' END,

          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%data engineering%' THEN 'data engineering' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%data pipeline%' THEN 'data pipeline' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%data warehouse%' THEN 'data warehouse' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%data lake%' THEN 'data lake' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%etl%' THEN 'etl' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%spark%' THEN 'apache spark' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%kafka%' THEN 'apache kafka' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%sql%' THEN 'sql' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%database%' THEN 'database' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%big data%' THEN 'big data' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%databricks%' THEN 'databricks' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%postgresql%' THEN 'postgresql' END,

          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%cloud computing%' THEN 'cloud computing' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%cloud architecture%' THEN 'cloud architecture' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%cloud security%' THEN 'cloud security' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%kubernetes%' THEN 'kubernetes' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%serverless%' THEN 'serverless' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%terraform%' THEN 'terraform' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%devops%' THEN 'devops' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%microservices%' THEN 'microservices' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%aws%' THEN 'aws' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%azure%' THEN 'azure' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%google cloud%' THEN 'google cloud' END,
          CASE WHEN LOWER(CONCAT_WS(' ', title, description)) LIKE '%docker%' THEN 'docker' END

        ), ''),

        -- Tier 2: Extract keywords from title and description
        NULLIF(
          LOWER(
            CONCAT_WS(', ',
              SLICE(
                FILTER(
                  TRANSFORM(
                    SPLIT(
                      LOWER(CONCAT_WS(' ', title, description)),
                      ' '
                    ),
                    w -> REGEXP_REPLACE(w, '[^a-z0-9]', '')
                  ),
                  w -> LENGTH(w) >= 4
                       AND w NOT IN (
                         'with','from','this','that','your','have',
                         'what','when','where','which','will','into',
                         'over','using','introduction','course','guide',
                         'tutorial','learn','basics','advanced','complete',
                         'master','part','step','about','they','their',
                         'there','these','those','then','than','also',
                         'more','some'
                       )
                ),
                1,
                5
              )
            )
          ),
          ''
        ),

        -- Tier 3: Category fallback
        LOWER(TRIM(category))

      ) AS list_of_keywords,

      TRIM(language) AS language,
      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,
      INITCAP(NULLIF(TRIM(difficulty_level), '')) AS difficulty_level,
      TRY_CAST(published_date AS TIMESTAMP) AS published_date,
      TRY_CAST(last_updated AS TIMESTAMP) AS last_updated,
      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.api_raw

    WHERE LOWER(TRIM(source)) = 'github'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
)

THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0



### Load YouTube Videos into Silver

This cell cleans, standardizes, generates keywords, deduplicates, and updates YouTube data from Bronze to Silver.

In [0]:
%sql

MERGE INTO edtech.silver.youtube_videos AS t

USING (

  WITH c AS (

    SELECT

      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,

      -- 3-tier keyword extraction: dictionary → title & description → category
      COALESCE(

        -- Tier 1: Dictionary matching
        NULLIF(CONCAT_WS(', ',

          CASE WHEN LOWER(title || ' ' || description) LIKE '%artificial intelligence%' THEN 'artificial intelligence' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%machine learning%' THEN 'machine learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%deep learning%' THEN 'deep learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%generative ai%' THEN 'generative ai' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%large language model%' THEN 'large language model' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%llm%' THEN 'llm' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%computer vision%' THEN 'computer vision' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%nlp%' THEN 'nlp' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%reinforcement learning%' THEN 'reinforcement learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%transformer%' THEN 'transformer' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%neural network%' THEN 'neural network' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%gpt%' THEN 'gpt' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%pytorch%' THEN 'pytorch' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%tensorflow%' THEN 'tensorflow' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%data engineering%' THEN 'data engineering' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data pipeline%' THEN 'data pipeline' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data warehouse%' THEN 'data warehouse' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data lake%' THEN 'data lake' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%etl%' THEN 'etl' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%spark%' THEN 'apache spark' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kafka%' THEN 'apache kafka' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%sql%' THEN 'sql' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%database%' THEN 'database' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%big data%' THEN 'big data' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%databricks%' THEN 'databricks' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%postgresql%' THEN 'postgresql' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud computing%' THEN 'cloud computing' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud architecture%' THEN 'cloud architecture' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud security%' THEN 'cloud security' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kubernetes%' THEN 'kubernetes' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%serverless%' THEN 'serverless' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%terraform%' THEN 'terraform' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%devops%' THEN 'devops' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%microservices%' THEN 'microservices' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%aws%' THEN 'aws' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%azure%' THEN 'azure' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%google cloud%' THEN 'google cloud' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%docker%' THEN 'docker' END

        ), ''),

        -- Tier 2: Extract keywords from title and description
        NULLIF(
          LOWER(
            CONCAT_WS(', ',
              SLICE(
                FILTER(
                  TRANSFORM(
                    SPLIT(
                      LOWER(CONCAT_WS(' ', title, description)),
                      ' '
                    ),
                    w -> REGEXP_REPLACE(w, '[^a-z0-9]', '')
                  ),
                  w -> LENGTH(w) >= 4
                       AND w NOT IN (
                         'with','from','this','that','your','have',
                         'what','when','where','which','will','into',
                         'over','using','introduction','course','guide',
                         'tutorial','learn','basics','advanced','complete',
                         'master','part','step','about','they','their',
                         'there','these','those','then','than','also',
                         'more','some'
                       )
                ),
                1,
                5
              )
            )
          ),
          ''
        ),

        -- Tier 3: Category fallback
        LOWER(TRIM(category))

      ) AS list_of_keywords,

      TRIM(language) AS language,
      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,

      INITCAP(
        NULLIF(TRIM(difficulty_level), '')
      ) AS difficulty_level,

      TRY_CAST(published_date AS TIMESTAMP) AS published_date,
      TRY_CAST(last_updated AS TIMESTAMP) AS last_updated,

      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.api_raw

    WHERE LOWER(TRIM(source)) = 'youtube'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
)

THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### Create Validation Views

This cell creates unified temporary views for Silver and Bronze data to compare and validate records.

In [0]:
-- Creates unified temporary views for Silver and Bronze data to compare and validate records.

CREATE OR REPLACE TEMP VIEW v_silver_all AS
SELECT 'blogs' AS tbl, content_id, url, topic, category, CAST(NULL AS STRING) AS difficulty_level, published_date, last_updated FROM edtech.silver.blogs
UNION ALL SELECT 'newsletters',      content_id, url, topic, category, NULL,             published_date, last_updated FROM edtech.silver.newsletters
UNION ALL SELECT 'coursera_courses', content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.coursera_courses
UNION ALL SELECT 'microsoft_learn',  content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.microsoft_learn
UNION ALL SELECT 'github_repos',     content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.github_repos
UNION ALL SELECT 'youtube_videos',   content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.youtube_videos;

-- Maps each Bronze row to its Silver table. Rows that match nothing get tbl = NULL.
CREATE OR REPLACE TEMP VIEW v_bronze_all AS
SELECT CASE LOWER(TRIM(content_type))
         WHEN 'article'    THEN 'blogs'
         WHEN 'newsletter' THEN 'newsletters' END AS tbl,
       content_id, title, url
FROM edtech.bronze.rss_raw
UNION ALL
SELECT CASE LOWER(TRIM(source))
         WHEN 'coursera'        THEN 'coursera_courses'
         WHEN 'microsoft learn' THEN 'microsoft_learn'
         WHEN 'github'          THEN 'github_repos'
         WHEN 'youtube'         THEN 'youtube_videos' END,
       content_id, title, url
FROM edtech.bronze.api_raw;

### Validate Silver vs Bronze

This cell compares valid Bronze records with Silver records and identifies any data gaps.

In [0]:
-- Compares valid Bronze records with Silver records and identifies data gaps.

SELECT b.tbl, 
       b.bronze_rows, 
       b.bronze_valid_ids, 
       s.silver_rows, 
       b.bronze_valid_ids - s.silver_rows AS unexplained_gap 
FROM ( 
  -- Counts total and valid Bronze records
  SELECT tbl, 
         COUNT(*) AS bronze_rows, 
         COUNT(DISTINCT CASE WHEN NULLIF(TRIM(content_id),'') IS NOT NULL 
                              AND NULLIF(TRIM(title),'')      IS NOT NULL 
                              AND NULLIF(TRIM(url),'')        IS NOT NULL 
                             THEN TRIM(content_id) END) AS bronze_valid_ids 
  FROM v_bronze_all GROUP BY tbl 
) b 
LEFT JOIN (
  -- Counts records in Silver
  SELECT tbl, COUNT(*) AS silver_rows 
  FROM v_silver_all GROUP BY tbl
) s 
  ON b.tbl = s.tbl 
ORDER BY b.tbl;

tbl,bronze_rows,bronze_valid_ids,silver_rows,unexplained_gap
blogs,565,565,565,0
coursera_courses,300,300,300,0
github_repos,300,300,300,0
microsoft_learn,300,300,300,0
newsletters,35,35,35,0
youtube_videos,300,300,300,0


### Check Data Quality

This cell checks the overall data quality of the Silver tables, including duplicates, bad URLs, and date issues.

In [0]:
-- Checks the overall data quality of Silver tables.
SELECT tbl,
       COUNT(*)                                                   AS `rows`,
       COUNT(*) - COUNT(DISTINCT content_id)                      AS duplicate_ids,
       COUNT(*) - COUNT(DISTINCT url)                             AS duplicate_urls,
       SUM(CASE WHEN url NOT LIKE 'http%' THEN 1 ELSE 0 END)      AS bad_urls,
       SUM(CASE WHEN published_date IS NULL THEN 1 ELSE 0 END)    AS null_published,
       SUM(CASE WHEN published_date > current_timestamp()
                  OR published_date < TIMESTAMP '2000-01-01'
                THEN 1 ELSE 0 END)                                AS out_of_range_dates
FROM v_silver_all
GROUP BY tbl
ORDER BY tbl;

tbl,rows,duplicate_ids,duplicate_urls,bad_urls,null_published,out_of_range_dates
blogs,565,0,0,0,0,0
coursera_courses,300,0,0,0,0,0
github_repos,300,0,0,0,0,0
microsoft_learn,300,0,0,0,0,0
newsletters,35,0,0,0,0,0
youtube_videos,300,0,0,0,0,0


### Analyze Data Distribution

This cell analyzes the distribution of topic, category, and difficulty levels across the Silver data.

In [0]:
-- Analyzes the distribution of topic, category, and difficulty levels.

SELECT 'topic' AS col, topic AS value, COUNT(*) AS n FROM v_silver_all GROUP BY topic
UNION ALL
SELECT 'category', category, COUNT(*) FROM v_silver_all GROUP BY category
UNION ALL
SELECT 'difficulty_level', difficulty_level, COUNT(*) FROM v_silver_all GROUP BY difficulty_level
ORDER BY col, n DESC;

col,value,n
category,Cloud Computing,700
category,Machine Learning,276
category,Artificial Intelligence,213
category,Data Engineering,180
category,Data,84
category,Generative AI,77
category,AI,59
category,Reinforcement Learning,55
category,Kubernetes,47
category,Natural Language Processing,42


### Check for NaN Keywords

This cell checks the row counts and whether any NaN values remain in the keywords column.

In [0]:
-- Check current row counts and whether NaN still exists
SELECT 'github_repos'    AS tbl, COUNT(*) AS rows, SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) AS nan_count FROM edtech.silver.github_repos
UNION ALL
SELECT 'microsoft_learn',        COUNT(*),         SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.microsoft_learn
UNION ALL
SELECT 'youtube_videos',         COUNT(*),         SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.youtube_videos;

tbl,rows,nan_count
github_repos,300,0
microsoft_learn,300,0
youtube_videos,300,0


### Final Row Counts

This cell shows the final row counts for all Silver tables.

In [0]:
SELECT 'blogs' AS tbl, COUNT(*) AS rows FROM edtech.silver.blogs
UNION ALL SELECT 'newsletters', COUNT(*) FROM edtech.silver.newsletters
UNION ALL SELECT 'coursera_courses', COUNT(*) FROM edtech.silver.coursera_courses
UNION ALL SELECT 'microsoft_learn', COUNT(*) FROM edtech.silver.microsoft_learn
UNION ALL SELECT 'github_repos', COUNT(*) FROM edtech.silver.github_repos
UNION ALL SELECT 'youtube_videos', COUNT(*) FROM edtech.silver.youtube_videos;

tbl,rows
blogs,565
newsletters,35
coursera_courses,300
microsoft_learn,300
github_repos,300
youtube_videos,300
